In [1]:
import polars as pl

In [3]:
data = pl.read_parquet("data/processed/all_features.parquet")

In [4]:
data.describe()

statistic,permno,date,prc,vol,ret,shrout,bidlo,askhi,symbol,ret_1d,z_close,z_volume,ma_ratio_5d,ma_ratio_10d,ma_ratio_20d,mom_1m,mom_3m,mom_6m,mom_12m,mom_12_1m,log_ret_cum,realized_vol_20d,realized_vol_60d,beta,idiosyncratic_vol,gsector,ggroup
str,f64,str,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str
"""count""",1.6774395e7,"""16774395""",1.654593e7,1.6545912e7,1.6541326e7,1.6774394e7,1.654593e7,1.654593e7,"""16774395""",1.6541326e7,1.6049548e7,1.442498e7,120544.0,15822.0,38969.0,1.6467807e7,1.6294416e7,1.603135e7,1.5520011e7,1.5506917e7,1.6541326e7,1.6467587e7,1.6302483e7,1.6302483e7,1.6059709e7,"""14353988""","""14353988"""
"""null_count""",0.0,"""0""",228465.0,228483.0,233069.0,1.0,228465.0,228465.0,"""0""",233069.0,724847.0,2.349415e6,1.6653851e7,1.6758573e7,1.6735426e7,306588.0,479979.0,743045.0,1.254384e6,1.267478e6,233069.0,306808.0,471912.0,471912.0,714686.0,"""2420407""","""2420407"""
"""mean""",53989.72036,"""2017-11-03 11:42:06.270000""",36.265367,1.3059e6,0.000491,123569.703373,36.394621,37.46296,null,0.000491,2.5264e-8,0.000001,-0.000395,0.000014,-0.000073,0.000007,-0.000002,1.7259e-7,0.000008,0.000011,0.000002,0.000008,0.000016,-0.000015,0.000032,null,null
"""std""",33237.197937,null,113.504404,7.1692e6,0.050218,455075.189065,111.856552,114.692982,null,0.050218,0.999757,0.999733,1.015374,1.045344,0.993899,0.999767,0.999763,0.999755,0.999754,0.999752,0.99976,0.999778,0.99978,0.999766,0.999804,null,null
"""min""",10001.0,"""2010-01-04 00:00:00""",-827.63,0.0,-0.954306,9.0,0.0001,0.0014,"""A""",-0.954306,-5.067069,-4.742237,-59.467217,-16.954657,-8.574171,-5.277223,-5.933237,-6.769398,-5.121256,-4.846092,-6.028819,-5.025202,-4.272382,-8.035163,-4.246907,"""10""","""1010"""
"""25%""",17266.0,"""2014-01-30 00:00:00""",4.56,43602.0,-0.013928,16353.0,4.9327,5.21,null,-0.013928,-0.551256,-0.590582,-0.388028,-0.407205,-0.41413,-0.602414,-0.65282,-0.678919,-0.700792,-0.698313,-0.65912,-0.655212,-0.671367,-0.644554,-0.677806,null,null
"""50%""",65008.0,"""2018-01-10 00:00:00""",14.98,222240.0,0.0,37857.0,14.99,15.56,null,0.0,-0.026991,-0.195448,-0.035692,0.002478,0.001869,-0.067825,-0.099559,-0.131895,-0.171552,-0.16614,0.079058,-0.205574,-0.189814,-0.035555,-0.200725,null,null
"""75%""",86965.0,"""2021-10-06 00:00:00""",38.01,841029.0,0.013165,91232.0,37.65,38.8,null,0.013165,0.521045,0.30441,0.375432,0.401446,0.391187,0.519249,0.541848,0.546654,0.55647,0.555179,0.707149,0.425188,0.479689,0.618219,0.478009,null,null
"""max""",93436.0,"""2024-12-31 00:00:00""",9924.40039,1.9237e9,39.725304,2.92064e7,9794.0,9964.76953,"""ZZ""",39.725304,5.08296,6.605526,13.553358,37.075163,8.830426,6.691857,6.053833,6.920173,8.562653,8.562179,7.327224,10.829866,10.457665,5.213817,9.961447,"""60""","""6020"""


In [8]:
from src.data.wrds_loader import WRDSDataLoader
import os 

os.environ["WRDS_USERNAME"]="niknatan"
os.environ["WRDS_PASSWORD"]="Nikita201001!"
with WRDSDataLoader() as loader:
    # 1. Check if comp.names has your tickers
    df = loader.conn.raw_sql("""
        SELECT DISTINCT tic FROM comp.names
        WHERE tic IN ('AAPL', 'MSFT', 'GOOGL', 'JPM')
    """)
    print("comp.names tickers:", df)

    # 2. Check what tables are in comp schema
    df2 = loader.conn.list_tables('comp')
    print("comp tables:", df2)

    # 3. Check actual fundq columns
    df3 = loader.conn.describe_table('comp', 'fundq')
    print("fundq schema:", df3)


Loading library list...
Done
comp.names tickers:      tic
0   AAPL
1  GOOGL
2    JPM
3   MSFT
comp tables: ['aco_amda', 'aco_imda', 'aco_indfnta', 'aco_indfntq', 'aco_indfntytd', 'aco_indsta', 'aco_indstq', 'aco_indstytd', 'aco_notesa', 'aco_notesq', 'aco_notessa', 'aco_notesytd', 'aco_pnfnda', 'aco_pnfndq', 'aco_pnfndytd', 'aco_pnfnta', 'aco_pnfntq', 'aco_pnfntytd', 'aco_transa', 'aco_transq', 'aco_transsa', 'aco_transytd', 'adsprate', 'asec_amda', 'asec_imda', 'asec_notesa', 'asec_notesq', 'asec_transa', 'asec_transq', 'bank_aacctchg', 'bank_adesind', 'bank_afnd1', 'bank_afnd2', 'bank_afnddc1', 'bank_afnddc2', 'bank_afntind', 'bank_funda', 'bank_funda_fncd', 'bank_fundq', 'bank_fundq_fncd', 'bank_iacctchg', 'bank_idesind', 'bank_ifndq', 'bank_ifndytd', 'bank_ifntq', 'bank_ifntytd', 'bank_names', 'bank_namesq', 'chars', 'co_aacctchg', 'co_aaudit', 'co_acthist', 'co_adesind', 'co_adjfact', 'co_afnd1', 'co_afnd2', 'co_afnddc1', 'co_afnddc2', 'co_afntind1', 'co_afntind2', 'co_ainvval', '

In [9]:
with WRDSDataLoader() as loader:
    # Check what's in wrds_ratios
    schema = loader.conn.describe_table('comp', 'wrds_ratios')
    print(schema[['name', 'comment']].to_string())

    # Check a sample row
    sample = loader.conn.raw_sql("""
        SELECT * FROM comp.wrds_ratios LIMIT 3
    """)
    print(sample.columns.tolist())
    print(sample.head())


Loading library list...
Done
Approximately 3495136 rows in comp.wrds_ratios.
               name                                         comment
0             gvkey                              Global Company Key
1             adate                                 Fiscal year end
2             qdate                              Fiscal quarter end
3       public_date                                            Date
4             capei          Shillers Cyclically Adjusted P/E Ratio
5                be                                            None
6                bm                                     Book/Market
7               evm                       Enterprise Value Multiple
8       pe_op_basic      Price/Operating Earnings (Basic, Excl. EI)
9         pe_op_dil    Price/Operating Earnings (Diluted, Excl. EI)
10           pe_exi                         P/E (Diluted, Excl. EI)
11           pe_inc                         P/E (Diluted, Incl. EI)
12               ps                    

In [10]:
with WRDSDataLoader() as loader:
    gvkeys = loader._get_gvkeys(['AAPL', 'MSFT', 'GOOGL'])
    print("GVKEYs:", gvkeys)

    if gvkeys:
        sample = loader.fetch_fundamentals(['AAPL', 'MSFT'], '2020-01-01', '2021-01-01')
        print(sample.shape)
        print(sample.columns.tolist())


Loading library list...
Done
GVKEYs: ['001690', '012141', '160329']


ProgrammingError: (psycopg2.errors.UndefinedColumn) column "seq" does not exist
LINE 2:             SELECT gvkey, rdq, atq, seq, niq, cshoq, prccq,
                                            ^

[SQL: 
            SELECT gvkey, rdq, atq, seq, niq, cshoq, prccq,
                   epspxq, opepsq, ceqq, txtq, xintq, saleq, cheq
            FROM comp.fundq
            WHERE gvkey IN ('001690','012141')
              AND rdq BETWEEN '2020-01-01' AND '2021-01-01'
              AND indfmt = 'INDL'
              AND datafmt = 'STD'
              AND popsrc = 'D'
            ORDER BY gvkey, rdq
        ]
(Background on this error at: https://sqlalche.me/e/20/f405)